# Zenoh — key expression, subscription, dan kueri

Dua sesi berbagi `VirtualZenohNetwork` dalam proses, jadi tidak ada yang membutuhkan pustaka native atau jaringan di sini. Untuk berbicara dengan peer sungguhan, hapus `UseVirtual(net)` dan pakai `Connect("tcp/host:7447")` atau `Listen("tcp/0.0.0.0:7447")`; pustaka native `iotcom_zenoh` ikut dalam paket.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.22.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.22.0-preview.1"
#r "nuget: IoTCom.Net.Adapters.Zenoh, 0.22.0-preview.1"

## Key expression
Key adalah chunk yang dipisah garis miring. `*` cocok dengan satu chunk, `**` dengan sejumlah chunk (termasuk nol), `$*` dengan bagian mana pun dari satu chunk.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Adapters.Zenoh;

foreach (var (pattern, key) in new[] { ("plant/*/temp", "plant/line1/temp"), ("plant/*/temp", "plant/a/b/temp"), ("plant/**", "plant/a/b/c"), ("sensor$*/temp", "sensor42/temp") })
    Console.WriteLine($"{pattern,-14} {(ZenohKeyExpr.Includes(pattern, key) ? "matches    " : "no match   ")} {key}");
Console.WriteLine($"plant/*/temp and plant/line1/** can both match one key: {ZenohKeyExpr.Intersects("plant/*/temp", "plant/line1/**")}");
Console.WriteLine($"valid \"plant//x\": {ZenohKeyExpr.IsValid("plant//x")}");

## Put, subscribe, dan delete
Subscription menerima setiap put dan delete yang key-nya cocok dengan ekspresinya, dari sesi lain.

In [ ]:
var net = new VirtualZenohNetwork();
var gateway = ZenohSession.Create(o => o.UseVirtual(net));
var sensor = ZenohSession.Create(o => o.UseVirtual(net));
await gateway.ConnectAsync();
await sensor.ConnectAsync();

var seen = new List<string>();
var subscription = gateway.Subscribe("plant/*/temp", s => { lock (seen) seen.Add($"{s.Kind} {s.Key} {s.Text}".TrimEnd()); });
await sensor.PutAsync("plant/line1/temp", "21.5");
await sensor.PutAsync("plant/line2/temp", "22.1");
await sensor.PutAsync("plant/line1/pressure", "1.8");   // other key: not delivered
await sensor.DeleteAsync("plant/line1/temp");
await Task.Delay(200);
lock (seen) foreach (var line in seen) Console.WriteLine(line);
subscription.Dispose();

## Memantau sebagai aliran
`WatchAsync` mengubah key expression menjadi `IAsyncEnumerable`.

In [ ]:
var cts = new CancellationTokenSource(TimeSpan.FromSeconds(10));
var watcher = Task.Run(async () =>
{
    var keys = new List<string>();
    await foreach (var s in gateway.WatchAsync("plant/**", cts.Token))
    {
        keys.Add(s.Key);
        if (keys.Count == 3) break;
    }
    return keys;
});
// The subscription is declared when enumeration starts, so keep publishing until the watcher has three samples.
for (var i = 0; i < 200 && !watcher.IsCompleted; i++) { await sensor.PutAsync("plant/line1/temp", $"v{i}"); await Task.Delay(10); }
Console.WriteLine(string.Join(", ", await watcher));

## Queryable dan get
Queryable menjawab permintaan `get`; `GetAsync` mengumpulkan jawaban dari setiap queryable yang cocok, termasuk jawaban error.

In [ ]:
var info = sensor.DeclareQueryable("plant/*/info", async q =>
{
    await q.ReplyAsync("plant/line1/info", $"state=running query={q.Parameters}");
    await q.ReplyErrorAsync("line2 offline");
});
foreach (var reply in await gateway.GetAsync("plant/*/info?detail=1"))
    Console.WriteLine(reply.IsError ? $"error: {reply.ErrorText}" : $"{reply.Sample!.Key}: {reply.Sample.Text}");
info.Dispose();

## Sesi hanya-baca
Put, delete, dan reply mengubah apa yang dilihat node lain. Sesi dengan `ReadOnly = true` menolaknya tetapi tetap bisa subscribe dan query.

In [ ]:
var viewer = ZenohSession.Create(o => { o.UseVirtual(net); o.ReadOnly = true; });
await viewer.ConnectAsync();
try { await viewer.PutAsync("plant/line1/setpoint", "99"); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
Console.WriteLine($"get still works: {(await viewer.GetAsync("plant/*/info", timeout: TimeSpan.FromMilliseconds(500))).Count} replies");
await viewer.DisposeAsync();
await sensor.DisposeAsync();
await gateway.DisposeAsync();

## Lebih lanjut
`iotcom zenoh sub "plant/**" --sim`, `iotcom zenoh get "plant/*/info" --sim`, `iotcom zenoh pub plant/line1/setpoint 42 --sim --allow-write`. Lihat `docs/id/protocols/zenoh.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*